In [1]:
import jax
import jax.numpy as jnp
from jax import random

jax.devices()

[CudaDevice(id=0)]

Random

In [1]:
key = random.key(seed=42)
print(key)

NameError: name 'random' is not defined

In [3]:
print(random.normal(key))
print(random.normal(key))

-0.18471177
-0.18471177


In [4]:
key, subkey = random.split(key)
print(key, subkey)

Array((), dtype=key<fry>) overlaying:
[2465931498 3679230171] Array((), dtype=key<fry>) overlaying:
[255383827 267815257]


Jax array

In [5]:
x = jnp.arange(5)
print(x)
print(type(x))
isinstance(x, jax.Array)

[0 1 2 3 4]
<class 'jaxlib.xla_extension.ArrayImpl'>


True

In [11]:
x.sharding, x.device

(SingleDeviceSharding(device=CudaDevice(id=0), memory_kind=device),
 CudaDevice(id=0))

Functions

In [35]:
from functools import partial
def linear_regression(x, w, b):
    return x * w + b
f = partial(linear_regression, w=jnp.ones(5), b=jnp.ones(5))
f

functools.partial(<function linear_regression at 0x78eaeabb9f80>, w=Array([1., 1., 1., 1., 1.], dtype=float32), b=Array([1., 1., 1., 1., 1.], dtype=float32))

In [16]:
x_0 = jnp.arange(5)
f(x_0)

Array([1., 2., 3., 4., 5.], dtype=float32)

Tracer and jaxpr (JAX exPRession)

In [28]:
from jax import jit
def f_impure(x):
    print(f'x={x}')
    print(f'type(x)={type(x)}')
    return x
result = f_impure(x_0)

x=[0 1 2 3 4]
type(x)=<class 'jaxlib.xla_extension.ArrayImpl'>


In [29]:
jax.make_jaxpr(f_impure)(x)

x=Traced<ShapedArray(int32[5])>with<DynamicJaxprTrace(level=1/0)>
type(x)=<class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>


{ lambda ; a:i32[5]. let  in (a,) }

JIT

In [39]:
x_1 = jnp.arange(10**3)
f = partial(linear_regression, w=jnp.ones(10**3), b=jnp.ones(10**3))
f(x_1).block_until_ready()
%timeit f(x_1).block_until_ready()

41.4 μs ± 3.74 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [40]:
f_jit = jit(f)
f_jit(x_1).block_until_ready()
%timeit f_jit(x_1).block_until_ready()

16.6 μs ± 199 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [45]:
'qw'.__hash__(), 'qw'.__hash__(), 'qw1'.__hash__(), 'qw1'.__hash__()

(1247026216386475699,
 1247026216386475699,
 -5177925807563804619,
 -5177925807563804619)

Vectorization

In [48]:
f = partial(linear_regression, w=jnp.arange(5), b=jnp.ones(5))
X = jnp.ones(shape = (2, 5))

f(X)

Array([[1., 2., 3., 4., 5.],
       [1., 2., 3., 4., 5.]], dtype=float32)

In [49]:
f_vmap = jax.vmap(f)
f_vmap(X)

Array([[1., 2., 3., 4., 5.],
       [1., 2., 3., 4., 5.]], dtype=float32)

In [52]:
def f(x,y):
    return x * y 
x_2 = jnp.arange(3)
f(x_2,x_2)

Array([0, 1, 4], dtype=int32)

In [53]:
f(X, X)

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]], dtype=float32)

In [55]:
f_vmap = jax.vmap(f)
f_vmap(X, X)

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]], dtype=float32)